# Process and plot Curie Temperature (Susceptibility-Temperature) Data 
Author: Peter Selkin  
Last Updated 8/20/2025  
  
This notebook analyzes and plots data from susceptibility-temperature experiments at hight temperature using an AGICO Kappabridge susceptibility meter like the one at Western Washington University. 

_Before running this notebook, make sure you have used the ['CUREVAL'](https://www.agico.cz/text/software/cureval/cureval.php) software to correct for the sample holder's susceptibility._  

This notebook makes extensive use of the [`pandas`](https://pandas.pydata.org/docs/getting_started/overview.html) package, especially the [`DataFrame`](https://pandas.pydata.org/docs/getting_started/intro_tutorials/02_read_write.html) data type. This is similar to the `data.frame` or _tibble_ (`tbl_df`) types in R. You might want to read up on DataFrames in order to get the most out of this notebook. You might also want to check out the first few chapters of Jake Vanderplas's [Python Data Science Handbook](https://github.com/jakevdp/PythonDataScienceHandbook).

This notebook also makes extensive use of Vanderplas's [`altair`](https://altair-viz.github.io/) package for plotting, somewhat analogous to `ggplot` in R. A large number of examples are available at <https://altair-viz.github.io/gallery/index.html>. These might give you some ideas about how to plot data from your experiments so you can communicate your interpretations. 

There are two basic techniques used to evaluate Curie/Neel temperatures (both of which are related to high-temperature transitions in magnetic mineralogy; we'll call both "Curie temperatures" for now, though Curie temperature is strictly for magnetite and Neel temperature is for hematite). Measuring susceptibility (usually called X or kappa) with temperature is one; the other uses strong fields and measures changes in Ms as a function of temperature. Chapter 6 of Tauxe's _Essentials of Paleomagnetism_ is a good start, but only covers results from strong-field Ms(T) experiments. Bruce Moskowitz's [_Hitchhiker's Guide to Rock Magnetism_](https://www.magneticmicrosphere.com/resources/hitchhiker_guide_to_magnetism.pdf) also has a very little bit. There are basically two categories of techniques that people use to find Curie temperatures (Tc): ones based on the slope of measured curves (_Essentials of Paleomagnetism_ uses this) and ones based on extrapolating linear portions of "inverse susceptibility" data to find an x-intercept (see for example Petrovsky and Kapicka, 2006). The latter is theoretically better for susceptibility, but the former is more frequently used. We'll use both here.

In [1]:
# Import packages: These are like "plugins" that provide specialized functions that aren't part of standard Python.
#
# Most of the packages used are installed with Anaconda. 
#
# I wrote Hysttools for managing hysteresis and IRM/DCD data. It is included with this notebook if you 
# downloaded it from Github. It relies on some functions from PmagPy, which you should have installed if
# you followed the setup directions on Github.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import re
import scipy.interpolate as spi
import scipy.optimize as spo
import scipy.stats as sps
import scipy.signal as spsig
import altair as alt
import curietools as ct
from pathlib import Path
from datetime import date

## Read Data

In [2]:
# Find the data files and identify the names of the samples in them.

# The following variables tell Python where to look for the files.
#repo=git.Repo('.', search_parent_directories=True)
path_root = Path(os.getcwd()).parent    # The "root" or base folder with the info for this project is one folder "up" 
                                        # from the folder where this notebook is. 
data_folder = path_root / 'data'
processed_folder = data_folder / 'processed'
kappa_folder = data_folder / 'raw'/'kappabridge'
image_folder = data_folder / 'images'

specimens=pd.read_excel(data_folder/'raw'/'metadata.xlsx')    # This file contains a list of files and the associated 
                                                              # sample names and info
sites=pd.read_excel(data_folder/'raw'/'sites.xlsx')
specimens=pd.merge(specimens,sites,on='site',how='left')      # Merge the sites with the specimens file: for each specimen, 
                                                              # add the corresponding info from the sites file based on which 
                                                              # site is listed in the "site" column 

specimens = specimens[~specimens['curie_file'].isna()]        # Only keep specimens that have Tc files listed

curie_pattern = re.compile('.*CUR$') # This is the suffix for a Curie temperature file
curie_list = list(filter(curie_pattern.match,os.listdir(kappa_folder))) # Make a list of all the Curie temperature files
yes_curie = specimens['curie_file'].isin(curie_list) # How many of the rows in the specimens data frame have files in the raw folder?
no_curie = ~specimens['curie_file'].isin(curie_list) # How many of the rows in the specimens data frame don't have files in the raw folder?
print("\n{} Curie temperature files to process.".format(np.shape(curie_list)[0]))
print(f"Matches:{sum(yes_curie)}\nNon-Matches:{sum(no_curie)}")

# Show the list of specimens (i.e. merged metadata and sites files)
display(specimens)


6 Curie temperature files to process.
Matches:6
Non-Matches:0


,irmdcd_file,hysu_file,curie_file,subsample,sample,site,season,year,sample_type,watershed,unit,grain_size,mass,Notes,site_name,lat,lon
14,2024-pr-20c.irmdcd,2024-pr-20c.hysu,24PR20C.CUR,2024-pr-20c,2024-pr-20,pr02,wi,2025.0,suspended,puyallup,NaN,z,0.1182,NaN,Puyallup River - Riverside Park,47.182609,-122.216639
23,2024-wr-20c.irmdcd,2024-wr-20c.hysu,24WR20C.CUR,2024-wr-20c,2024-wr-20,wr01,wi,2025.0,suspended,white,NaN,z,0.1026,NaN,White River at 24th Street,47.235415,-122.236326
24,2024-wr-21.irmdcd,2024-wr-21.hysu,24WR21.CUR,2024-wr-21,2024-wr-21,wr01,wi,2025.0,bed,white,NaN,z,0.1503,NaN,White River at 24th Street,47.235415,-122.236326
28,2025-gp-01a-01.irmdcd,2025-gp-01a-01.hysu,25GP01A1.CUR,2025-gp-01a-01,2025-gp-01a,gp01,NaN,NaN,source,NaN,Qob-g?,z,0.1230,Pleistocene unit unknown - inferred based on T...,NaN,NaN,NaN
30,2025-mm-10a-2.irmdcd,2025-mm-10a-2.hysu,25MM10A2.CUR,2025-mm-10a-02,2025-mm-10a,mm10,NaN,NaN,source,NaN,Qos,z,0.1096,NaN,NaN,NaN,NaN
31,2025-mm-11a1.irmdcd,2025-mm-11a-1.hysu,25MM11A1.CUR,2025-mm-11a-01,2025-mm-11a,mm11,NaN,NaN,source,NaN,Qos,z,0.1376,NaN,NaN,NaN,NaN


## Process Data

In [5]:
# This is where we process the data in the Curie temperature files.
# It takes A WHILE...
# Don't worry about the red warnings.

file_skip=[] # If there are any data files you want to skip, add the names of the files to this list. 
curie_columns=['temperature','raw_susceptibility','corrected_susceptibility','normalized_susceptibility','bulk_susceptibility','ferrt','ferrb','time','holder_susceptibility']

# Iterate through each row of the list of specimens...
for ix,row in specimens.iterrows():
     # First we work on the IRM/DC demagnetization data
    file_path = kappa_folder / row['curie_file']
    print('Working on {}\n{}\n'.format(row['subsample'],file_path))
    curie_data=ct.read_curie(file_path)
    
    results=ct.process_curie(curie_data, smooth=10, name=row['subsample'])
    

    print(f'{row.subsample}\nInverse susceptibility method Tc\n',\
          f'\tHeating: {results.Tc_inverse_heating}\n',\
          f'\tCooling: {results.Tc_inverse_cooling}')
    specimens.loc[ix,'Tc_inverse_heating']=results['Tc_inverse_heating']
    specimens.loc[ix,'Tc_inverse_cooling']=results['Tc_inverse_cooling']
    specimens.loc[ix,'Tc_derivative_heating']=results['Tc_derivative_heating']
    specimens.loc[ix,'Tc_derivative_cooling']=results['Tc_derivative_cooling']

Working on 2024-pr-20c
C:\Users\paselkin\Dropbox\Research\Glaciers2Bay\glacier-bay-uwt-student-projects\data\raw\kappabridge\24PR20C.CUR



alt.LayerChart(...)

Heating min T, max T:580,625
Cooling min T, max T:580,625


alt.HConcatChart(...)

Tc based on dK/dT heating, cooling:587.9, 582.9
2024-pr-20c
Inverse susceptibility method Tc
 	Heating: 577.4867303500243
 	Cooling: 577.9255153915333
Working on 2024-wr-20c
C:\Users\paselkin\Dropbox\Research\Glaciers2Bay\glacier-bay-uwt-student-projects\data\raw\kappabridge\24WR20C.CUR



alt.LayerChart(...)

KeyboardInterrupt: Interrupted by user

In [ ]:
specimens.to_csv(processed_folder/'curie_stats.csv') # Save it all as a csv file (can open in Excel)